# Retry & Fallback

### Setup

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv()

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

### Define Tools (Primary + Fallback)

In [2]:
# Primary Tool (Fails on Purpose)
def structured_json_summarizer(text: str) -> dict:
    raise RuntimeError("Schema validation failed")

In [3]:
# Fallback Tool (Always Works)
def simple_text_summarizer(text: str) -> str:
    return {
        "summary": text[:100],
        "confidence": "low"
    }


### Register Tools with OpenAI (Function Calling Schema)

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "structured_json_summarizer",
            "description": "Generate a structured JSON summary",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string"}
                },
                "required": ["text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "simple_text_summarizer",
            "description": "Generate a simple fallback summary",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string"}
                },
                "required": ["text"]
            }
        }
    }
]

### User Prompt (Natural Language)

In [5]:
user_prompt = """
Convert the following text into a structured JSON summary.

Text:
Agentic AI systems autonomously plan and act using tools in multi-step workflows.
"""

### First Attempt — Model Chooses Primary Tool

In [6]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": user_prompt}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
tool_call = message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

### Execute Primary Tool (Failure)

In [7]:
try:
    result = structured_json_summarizer(**args)
except Exception as e:
    print("❌ Primary tool failed:", e)

❌ Primary tool failed: Schema validation failed


### Retry with Clarified Instructions

In [8]:
retry_prompt = """
The previous attempt failed.
Please generate a simpler summary in JSON format.
"""

retry_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": retry_prompt},
        {"role": "user", "content": user_prompt}
    ],
    tools=tools,
    tool_choice="auto"
)

retry_message = retry_response.choices[0].message
retry_tool_call = retry_message.tool_calls[0]
retry_args = json.loads(retry_tool_call.function.arguments)

### Automatic Fallback Tool Execution

In [9]:
fallback_result = simple_text_summarizer(**retry_args)

print("✅ Fallback Tool Output:")
print(json.dumps(fallback_result, indent=2))

✅ Fallback Tool Output:
{
  "summary": "Agentic AI systems autonomously plan and act using tools in multi-step workflows.",
  "confidence": "low"
}
